# Tracing LLM calls with Langfuse

The City Museum's ticket bot answers a group-discount question wrong since the nightly re-index. In this notebook
you run the same bot with the real Langfuse Python SDK, print the spans of one request as a tree, find the step
that brought in the wrong fact, compare with yesterday's trace, and debug a second bug from its trace.

About 10 minutes. Run the cells from top to bottom. Everything runs on a free CPU runtime, with no account and no key.

Two things are different from production, and both are marked in the code:
- **No Langfuse server.** The SDK keeps the spans in memory (its `span_exporter` option), and we print them.
- **No live model.** The bot's model answers were recorded when the lesson was made. The bot still builds the
  prompt and calls `call_model`, which looks up the recorded answer for exactly that prompt.

## 1. Install the Langfuse SDK

The next cell installs the Langfuse version the lesson ran (5.2.0). It brings OpenTelemetry with it.

**Look for:** no error. pip may print a few lines about other packages; you can ignore them.

In [1]:
%pip install -q langfuse==5.2.0

Note: you may need to restart the kernel to use updated packages.


## 2. The museum's help pages and the old price list

`WEBSITE` holds the ten help pages, one paragraph per topic. `ARCHIVE` holds the text of an old 2024 price list
PDF: a header, the 2024 group rule, and one line per opening day of 2024. A PDF-to-text tool gives no blank lines.

**Look for:** the current group rule (15 or more people, 20% off) and the 2024 rule (10 or more adults, 15% off).

In [2]:
import datetime

WEBSITE = {
    "tickets.md": """Tickets: an adult ticket costs 16 euro. Children under 6 go free. Children from 6 to 17 pay 8 euro. Students and seniors over 65 pay 11 euro with a card or ID. Tickets are valid for one day and include all permanent galleries.

Buying tickets: you can buy tickets online or at the front desk. Online tickets have a QR code that is scanned at the entrance. Front desk tickets are paper.""",

    "groups.md": """Group tickets: from 1 March 2026, groups of 15 or more people get 20% off the adult price. Smaller groups pay the normal price per person. Book at least 48 hours ahead with the group form; the group leader gets one free ticket for every 15 paid tickets.

Group arrival: groups enter by the north door and keep together in the first gallery. Bags larger than a backpack go to the cloakroom.""",

    "schools.md": """School visits: school classes visit free on weekday mornings from 9:00 to 12:00 when a teacher books ahead. One teacher is needed for every 10 pupils. Schools on weekends pay the normal group price.""",

    "hours.md": """Opening hours: the museum is open Tuesday to Sunday from 10:00 to 18:00, and until 21:00 on Thursdays. It is closed on Mondays, on 25 December and on 1 January. The last entry is one hour before closing.""",

    "tours.md": """Guided tours: a guide can lead your group for 60 minutes for 90 euro, on top of the tickets. Tours run in English, German and French. Ask for a tour on the group form at least one week ahead.""",

    "access.md": """Accessibility: every gallery has step-free access and lifts. Wheelchairs can be borrowed at the cloakroom for free. A carer of a disabled visitor enters free.""",

    "membership.md": """Membership: a yearly membership costs 45 euro and gives free entry for one adult on every visit, plus 10% off in the shop. A family membership costs 90 euro and covers two adults and their children.""",

    "refunds.md": """Refunds: online tickets can be refunded up to 24 hours before the visit date. Group bookings can be cancelled free up to 48 hours ahead; after that, half of the group price is charged.""",

    "cafe.md": """Cafe and shop: the cafe on the ground floor is open during museum hours. Groups can pre-order lunch boxes on the group form. The shop sells books, posters and toys.""",

    "events.md": """Evening events: on Thursday evenings there are talks and concerts in the main hall. Event tickets are sold separately and do not include the galleries.""",
}

def price_list_2024_pdf_text():
    """The text of the old 2024 price list PDF: a header, the 2024 group rule, and one row per day of 2024."""
    lines = [
        "CITY MUSEUM PRICE LIST 2024",
        "Valid 1 January 2024 to 31 December 2024",
        "Group discount: groups of 10 or more adults get 15% off the adult price. No booking needed.",
        "Date | Adult | Child 6-17 | Student/Senior | Group adult (10+)",
    ]
    day = datetime.date(2024, 1, 1)
    while day.year == 2024:
        if day.weekday() != 0:  # closed on Mondays
            weekend = day.weekday() >= 5
            adult = 14.0 if not weekend else 15.0
            lines.append(
                f"{day.strftime('%a %d %b %Y')} | Adult {adult:.2f} | Child {adult/2:.2f} | "
                f"Student/Senior {adult*0.7:.2f} | Group adult (10+) {adult*0.85:.2f}"
            )
        day += datetime.timedelta(days=1)
    return "\n".join(lines)

ARCHIVE = {"archive/price-list-2024.pdf": price_list_2024_pdf_text()}

print("help pages:", len(WEBSITE))
print("current rule:", WEBSITE["groups.md"].split(". Book")[0])
pdf = ARCHIVE["archive/price-list-2024.pdf"]
print("2024 rule:   ", pdf.splitlines()[2])
print("PDF text:", len(pdf.splitlines()), "lines,", len(pdf), "characters")

help pages: 10
current rule: Group tickets: from 1 March 2026, groups of 15 or more people get 20% off the adult price. Smaller groups pay the normal price per person
2024 rule:    Group discount: groups of 10 or more adults get 15% off the adult price. No booking needed.
PDF text: 317 lines, 28810 characters


## 3. The index and the search

Every night a job splits each document on blank lines into chunks. `score` counts how many of the question's
words a chunk contains. It does not divide by the chunk's length, so a huge chunk can win just by being huge.
Yesterday the job read only the website. Last night it also read the archive folder.

**Look for:** 12 chunks yesterday, 13 today. The new chunk is the whole PDF in one piece.

In [3]:
import re

def build_index(docs):
    chunks = []
    for source, text in docs.items():
        for part in text.split("\n\n"):          # split on blank lines
            chunks.append({"source": source, "text": part.strip()})
    return chunks

STOP = set("we are do get a and how is it the on to of in for you with can".split())
def words(text):
    return [w.rstrip("s") for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

def score(question, chunk):
    q = set(words(question))
    return sum(1 for w in words(chunk["text"]) if w in q)   # raw count, no length normalisation

def make_index(run):
    docs = dict(WEBSITE)
    if run == "today":
        docs.update(ARCHIVE)                           # the nightly job now also reads /archive
    return build_index(docs)

for run in ["yesterday", "today"]:
    idx = make_index(run)
    big = max(idx, key=lambda c: len(c["text"]))
    print(f"{run:<9}  chunks: {len(idx)}   biggest: {big['source']} ({len(big['text']):,} characters)")

yesterday  chunks: 12   biggest: groups.md (253 characters)
today      chunks: 13   biggest: archive/price-list-2024.pdf (28,810 characters)


## 4. The recorded model answers

When the lesson was made, the bot sent its prompts to a small hosted model (`google/gemini-2.5-flash-lite`,
temperature 0). The three answers and their token counts are below, copied from the recordings. Each one is
filed under a fingerprint of the exact prompt that was sent: the first 16 characters of a SHA-256 hash of the
messages.

`call_model` makes the same fingerprint from the prompt the bot just built. If it matches, it returns the recorded
answer. If the prompt is different in any way, there is no recorded answer, and it says so.

A hosted model took between 1.4 and 3.8 seconds to answer in the lesson's runs. A replay takes no time, so
`call_model` waits `MODEL_WAIT_S` seconds to stand in for the model.

**Look for:** three recorded answers. Only one of them is right for 12 adults.

In [4]:
import hashlib, json, time

MODEL = "google/gemini-2.5-flash-lite"
MODEL_WAIT_S = 2.0   # stands in for the hosted model's answer time (1.4 to 3.8 s in the lesson's runs)

# Recorded answers, filed under a fingerprint of the exact messages that were sent.
RECORDED = {
    "08f4b790bda4f680": {"recording": "bot-yesterday", "model": "google/gemini-2.5-flash-lite", "recorded": "2026-11-13",
        "usage": {"prompt_tokens": 282, "completion_tokens": 17},
        "text": "Smaller groups pay the normal price per person. You will not receive a group discount."},
    "9d8323ec28ab6c7f": {"recording": "bot-today", "model": "google/gemini-2.5-flash-lite", "recorded": "2026-11-13",
        "usage": {"prompt_tokens": 15447, "completion_tokens": 32},
        "text": "Yes, you get a group discount. For groups of 10 or more adults, the price is 12.75 per person on Saturdays."},
    "d9c62a60762fca8a": {"recording": "bot-offbyone", "model": "google/gemini-2.5-flash-lite", "recorded": "2026-11-13",
        "usage": {"prompt_tokens": 207, "completion_tokens": 20},
        "text": "The context does not mention a group discount for Saturdays. Schools on weekends pay the normal group price."},
}

def fingerprint(messages):
    return hashlib.sha256(json.dumps(messages).encode()).hexdigest()[:16]

def call_model(messages):
    # Replay instead of a live call: same input (the messages), same output shape as the API.
    time.sleep(MODEL_WAIT_S)
    rec = RECORDED.get(fingerprint(messages))
    if rec is None:
        return {"model": MODEL, "usage": {"prompt_tokens": 0, "completion_tokens": 0},
                "text": "(no recorded answer for this prompt)"}
    return {"model": rec["model"], "usage": rec["usage"], "text": rec["text"]}

for key, rec in RECORDED.items():
    print(f"{rec['recording']:<14} {key}  input {rec['usage']['prompt_tokens']:>6,} tokens  ->  {rec['text']}")

bot-yesterday  08f4b790bda4f680  input    282 tokens  ->  Smaller groups pay the normal price per person. You will not receive a group discount.
bot-today      9d8323ec28ab6c7f  input 15,447 tokens  ->  Yes, you get a group discount. For groups of 10 or more adults, the price is 12.75 per person on Saturdays.
bot-offbyone   d9c62a60762fca8a  input    207 tokens  ->  The context does not mention a group discount for Saturdays. Schools on weekends pay the normal group price.


## 5. Start Langfuse, keeping the spans in memory

In production you give `Langfuse()` your keys and it sends spans to the Langfuse server. Here we hand it an
OpenTelemetry `InMemorySpanExporter` instead, so the spans stay in this notebook. The keys are placeholders;
nothing is sent anywhere.

`run_bot` builds the right index, asks the question once, calls `langfuse.flush()` so every span is finished,
and returns the spans. `show_tree` prints one line per span: when it started, how long it took, its type, its
name and its parent. `attr` reads one field that the SDK stored on a span.

**Look for:** no error. Nothing is printed yet.

In [5]:
import random
random.seed(0)   # OpenTelemetry makes trace ids with Python's random module

from langfuse import Langfuse, observe, get_client
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

exporter = InMemorySpanExporter()
Langfuse(public_key="pk-lf-local", secret_key="sk-lf-local", span_exporter=exporter)
langfuse = get_client()

QUESTION = "We are 12 adults coming on Saturday. Do we get a group discount, and how much is it?"
RUN, INDEX = "yesterday", make_index("yesterday")

def run_bot(run):
    global RUN, INDEX
    RUN, INDEX = run, make_index(run)
    exporter.clear()
    answer = answer_question(QUESTION)
    langfuse.flush()
    return answer, sorted(exporter.get_finished_spans(), key=lambda s: s.start_time)

def show_tree(spans):
    t0 = spans[0].start_time
    names = {s.context.span_id: s.name for s in spans}
    print("   start      took  type        name             parent           trace id")
    for s in spans:
        parent = names.get(s.parent.span_id) if s.parent else None
        print(f"{(s.start_time - t0) / 1e6:>8.1f}  {(s.end_time - s.start_time) / 1e6:>8.1f}  "
              f"{s.attributes.get('langfuse.observation.type'):<11} {s.name:<16} {str(parent):<16} "
              f"{format(s.context.trace_id, '032x')[:8]}")
    print("(times in ms)")

def attr(spans, name, field):
    s = next(s for s in spans if s.name == name)
    v = s.attributes.get("langfuse.observation." + field)
    try:
        return json.loads(v)
    except (TypeError, ValueError):
        return v

## 6. The bot, with `@observe`

This is the lesson's code. `@observe()` turns each call of a function into a span. Its arguments become the
span's input and its return value the span's output. `as_type="retriever"` and `as_type="generation"` set the
span's type; a generation can also carry a model name and token usage, which `generate` hands over with
`update_current_generation`.

`build_prompt` has a switch for the third run, `offbyone`: a small code change that skips the first chunk.

**Look for:** no output. The functions are defined, not run.

In [6]:
@observe()
def answer_question(question):
    chunks = retrieve(question, top_k=4)
    prompt = build_prompt(question, chunks)
    return generate(prompt)

@observe(as_type="retriever")
def retrieve(question, top_k):
    ranked = sorted(INDEX, key=lambda c: score(question, c), reverse=True)
    langfuse.update_current_span(metadata={"index_chunks": len(INDEX)})
    return [{"source": c["source"], "score": score(question, c), "chars": len(c["text"]), "text": c["text"]}
            for c in ranked[:top_k]]

@observe()
def build_prompt(question, chunks):
    if RUN == "offbyone":
        chunks = chunks[1:]                        # the bug: meant to skip a header row
    context = "\n\n".join(f"[{c['source']}]\n{c['text']}" for c in chunks)
    return [
        {"role": "system", "content": "You answer visitors' questions for the City Museum ticket desk. "
                                      "Use only the context. If the context does not answer the question, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

@observe(as_type="generation")
def generate(prompt):
    reply = call_model(prompt)
    langfuse.update_current_generation(model=reply["model"], usage_details={
        "input": reply["usage"]["prompt_tokens"], "output": reply["usage"]["completion_tokens"]})
    return reply["text"]

## 7. Today's bad answer, as a trace

The next cell asks the question with today's index (the one with the PDF) and prints the spans.

**Look for:** four spans. `answer_question` has no parent; the other three have `answer_question` as parent,
and all four share one trace id. The types are `span`, `retriever`, `span`, `generation`. `retrieve` and
`build_prompt` take a few milliseconds or less. `generate` takes nearly all of the time: here that is the
2-second wait that stands in for the model. The answer is the wrong one from the lesson (12.75 per person).

In [7]:
answer, today = run_bot("today")
print("answer:", answer)
print()
show_tree(today)

answer: Yes, you get a group discount. For groups of 10 or more adults, the price is 12.75 per person on Saturdays.

   start      took  type        name             parent           trace id
     0.0    2007.8  span        answer_question  None             e3e70682
     0.2       2.9  retriever   retrieve         answer_question  e3e70682
     3.3       0.2  span        build_prompt     answer_question  e3e70682
     3.7    2004.1  generation  generate         answer_question  e3e70682
(times in ms)


## 8. Open the spans

A trace's time shows where the work happened. Its inputs and outputs show why the answer is wrong. The next
cell prints what the SDK stored on each step of today's trace.

**Look for:** retrieve's first chunk is `archive/price-list-2024.pdf`, 28,810 characters, scored 1,061 against
6 for the current page. build_prompt passes the same four chunks on, in the same order. generate read 15,447
input tokens. The fingerprint matched the recording `bot-today`, so the prompt is exactly the one the lesson sent.

In [8]:
print("retrieve  metadata index_chunks =", attr(today, "retrieve", "metadata.index_chunks"))
for i, c in enumerate(attr(today, "retrieve", "output"), 1):
    print(f"  {i}. {c['source']:<28} score {c['score']:>5,}   {c['chars']:>6,} characters")

messages = attr(today, "build_prompt", "output")
user = messages[1]["content"]
print("build_prompt  chunks in the prompt:", re.findall(r"\[([^\]]+)\]\n", user))
print("              user message:", f"{len(user):,}", "characters")

print("generate  model:", attr(today, "generate", "model.name"))
print("          usage_details:", attr(today, "generate", "usage_details"))
print("          recording used:", RECORDED.get(fingerprint(messages), {}).get("recording"))

retrieve  metadata index_chunks = 13
  1. archive/price-list-2024.pdf  score 1,061   28,810 characters
  2. groups.md                    score     6      253 characters
  3. groups.md                    score     2      134 characters
  4. schools.md                   score     2      198 characters
build_prompt  chunks in the prompt: ['archive/price-list-2024.pdf', 'groups.md', 'groups.md', 'schools.md']
              user message: 29,573 characters
generate  model: google/gemini-2.5-flash-lite
          usage_details: {'input': 15447, 'output': 32}
          recording used: bot-today


## 9. Compare with yesterday's trace

Same question, same code, same prompt and same model, but yesterday's index (no PDF).

**Look for:** `index_chunks` is 12, not 13. retrieve's first chunk is `groups.md`. generate read 282 input
tokens, not 15,447: about 55 times fewer. The answer is right: 12 adults pay the normal price.

In [9]:
answer, yesterday = run_bot("yesterday")
print("answer:", answer)
print()
show_tree(yesterday)
print()
for name, spans in [("yesterday", yesterday), ("today", today)]:
    first = attr(spans, "retrieve", "output")[0]
    tokens = attr(spans, "generate", "usage_details")["input"]
    print(f"{name:<9}  index_chunks {attr(spans, 'retrieve', 'metadata.index_chunks'):>2}   "
          f"first chunk {first['source']:<28} input tokens {tokens:>6,}")
ratio = attr(today, "generate", "usage_details")["input"] / attr(yesterday, "generate", "usage_details")["input"]
print(f"today's prompt is {ratio:.1f} times larger")

answer: Smaller groups pay the normal price per person. You will not receive a group discount.

   start      took  type        name             parent           trace id
     0.0    2011.5  span        answer_question  None             f7c1bd87
     0.4       0.9  retriever   retrieve         answer_question  f7c1bd87
     1.4       0.2  span        build_prompt     answer_question  f7c1bd87
     1.8    2009.6  generation  generate         answer_question  f7c1bd87
(times in ms)

yesterday  index_chunks 12   first chunk groups.md                    input tokens    282
today      index_chunks 13   first chunk archive/price-list-2024.pdf  input tokens 15,447
today's prompt is 54.8 times larger


## 10. Broken on purpose: a second wrong answer

A week later the PDF is gone, but a small code change went in: `build_prompt` now skips the first chunk
(`chunks = chunks[1:]`, meant to skip a header row that does not exist). Read the trace and compare what goes
into `build_prompt` with what comes out.

**Look for:** a different wrong answer ("The context does not mention a group discount..."). retrieve returns
four chunks with `groups.md` first. build_prompt's prompt holds only three, and the group rule is the one missing.
The model followed "use only the context": it never saw the rule.

In [10]:
answer, offbyone = run_bot("offbyone")
print("answer:", answer)
print()
show_tree(offbyone)
print()
print("retrieve returned:       ", [c["source"] for c in attr(offbyone, "retrieve", "output")])
user = attr(offbyone, "build_prompt", "output")[1]["content"]
print("build_prompt passed on:  ", re.findall(r"\[([^\]]+)\]\n", user))
print("group rule in the prompt:", "15 or more people" in user)

answer: The context does not mention a group discount for Saturdays. Schools on weekends pay the normal group price.

   start      took  type        name             parent           trace id
     0.0    2005.9  span        answer_question  None             1846d424
     0.2       0.4  retriever   retrieve         answer_question  1846d424
     0.6       0.1  span        build_prompt     answer_question  1846d424
     0.7    2005.1  generation  generate         answer_question  1846d424
(times in ms)

retrieve returned:        ['groups.md', 'groups.md', 'schools.md', 'tours.md']
build_prompt passed on:   ['groups.md', 'schools.md', 'tours.md']
group rule in the prompt: False


## 11. Why the spans form one tree

A decorated function called while another span is open becomes its child, and shares its trace id. The SDK is
built on OpenTelemetry, which keeps the current span in the running code's context. The next cell calls
`generate` on its own, outside `answer_question`.

**Look for:** one span, with parent `None` and a new trace id, different from the trace id in step 10.

In [11]:
exporter.clear()
generate([{"role": "user", "content": "Hello"}])
langfuse.flush()
show_tree(exporter.get_finished_spans())
print("trace id in step 10:", format(offbyone[0].context.trace_id, "032x")[:8])

   start      took  type        name             parent           trace id
     0.0    2002.6  generation  generate         None             259f4329
(times in ms)
trace id in step 10: 1846d424


## Now change it

Three small experiments. Each cell has one line marked `# <- change this`. Change it and run the cell again.

### Experiment 1: stop storing the chunks

The retrieve span stores every chunk it returns: almost 30,000 characters per request today. If the index held
visitors' names, they would all be stored too. This cell redefines `retrieve` with `capture_output=CAPTURE`.
When capture is off, it stores only a short summary (source and score) with `update_current_span`.

**What should change:** with `CAPTURE = False`, retrieve's stored output is under 200 characters instead of
almost 30,000, and the answer is the same. Set it to `True` to see the full size again. build_prompt and
generate still store the chunk text in their own input and output, so in a real app you would switch capture
off (or mask) there too.

In [12]:
CAPTURE = False  # <- change this (True stores every chunk again)

@observe(as_type="retriever", capture_output=CAPTURE)
def retrieve(question, top_k):
    ranked = sorted(INDEX, key=lambda c: score(question, c), reverse=True)
    langfuse.update_current_span(metadata={"index_chunks": len(INDEX)})
    result = [{"source": c["source"], "score": score(question, c), "chars": len(c["text"]), "text": c["text"]}
              for c in ranked[:top_k]]
    if not CAPTURE:
        langfuse.update_current_span(output=[{"source": c["source"], "score": c["score"]} for c in result])
    return result

answer, spans = run_bot("today")
stored = next(s for s in spans if s.name == "retrieve").attributes.get("langfuse.observation.output")
print("answer:", answer)
print(f"retrieve output stored: {len(stored):,} characters")
print(stored[:200])

answer: Yes, you get a group discount. For groups of 10 or more adults, the price is 12.75 per person on Saturdays.
retrieve output stored: 170 characters
[{"source": "archive/price-list-2024.pdf", "score": 1061}, {"source": "groups.md", "score": 6}, {"source": "groups.md", "score": 2}, {"source": "schools.md", "score": 2}]


### Experiment 2: fix the bug from step 10

This cell redefines `build_prompt` with the slice on its own marked line and runs the `offbyone` version again.

**What should change:** with `chunks = chunks[1:]` you get the same wrong answer as in step 10. Change the line to
`chunks = chunks` (or delete the slice) and the group rule is back in the prompt. The prompt is now exactly
yesterday's prompt, so its fingerprint matches the recording `bot-yesterday`, and you get the right answer.

In [13]:
@observe()
def build_prompt(question, chunks):
    if RUN == "offbyone":
        chunks = chunks[1:]  # <- change this (try: chunks = chunks)
    context = "\n\n".join(f"[{c['source']}]\n{c['text']}" for c in chunks)
    return [
        {"role": "system", "content": "You answer visitors' questions for the City Museum ticket desk. "
                                      "Use only the context. If the context does not answer the question, say so."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

answer, spans = run_bot("offbyone")
messages = attr(spans, "build_prompt", "output")
print("chunks in the prompt:", re.findall(r"\[([^\]]+)\]\n", messages[1]["content"]))
print("recording used:      ", RECORDED.get(fingerprint(messages), {}).get("recording"))
print("answer:", answer)

chunks in the prompt: ['groups.md', 'schools.md', 'tours.md']
recording used:       bot-offbyone
answer: The context does not mention a group discount for Saturdays. Schools on weekends pay the normal group price.


### Experiment 3: your own question to the search

This cell calls `retrieve` on its own with today's index (with the PDF), so it becomes a trace of its own with one
span. It prints the four chunks it returns. No model is called.

**What should change:** for the question below, the 2024 PDF comes first with a score of 314, far above
`tickets.md`, which holds this year's prices: the PDF has a "Student/Senior" price on every one of its 313 lines,
and `score` counts every match. Now try `"Until what time is the museum open on Thursday?"`: `hours.md` comes
first and the PDF drops to 4th. The PDF wins every question that uses a word it repeats (student, child, adult,
group). This is why the fix is to keep the archive out of the index, or split such a file into small chunks.

In [14]:
my_question = "How much does a student ticket cost on Saturday?"  # <- change this

INDEX = make_index("today")
exporter.clear()
for i, c in enumerate(retrieve(my_question, top_k=4), 1):
    print(f"{i}. {c['source']:<28} score {c['score']:>4}   {c['chars']:>6,} characters")
langfuse.flush()
print()
show_tree(exporter.get_finished_spans())

1. archive/price-list-2024.pdf  score  314   28,810 characters
2. tickets.md                   score    5      226 characters
3. tickets.md                   score    4      157 characters
4. groups.md                    score    3      253 characters

   start      took  type        name             parent           trace id
     0.0       3.8  retriever   retrieve         None             ea7e9d49
(times in ms)


## What you should see

- Step 7: the printed tree has 4 spans: answer_question at the top, and retrieve, build_prompt, generate under it, all with the same trace id.
- Step 7: retrieve's type is `retriever` and generate's is `generation`; the other two are `span`.
- Step 8: with the 2024 PDF in the index, retrieve's first chunk is archive/price-list-2024.pdf, about 28,800 characters long.
- Step 9: generate's usage_details input is about 55 times larger with the PDF than without it (15,447 against 282 tokens).
- Step 7: retrieve and build_prompt take a few milliseconds or less; generate takes nearly all of the request's time (here the replay's 2-second wait stands in for the model).